Implement the Monte Carlo control algorithm in Python and apply it to a simple environment.

In [5]:
import numpy as np
import random
from collections import defaultdict

In [6]:
class SimpleGridWorld:
    def __init__(self):
        self.states = range(6)
        self.actions = [0, 1, 2, 3]
        self.goal_state = 0

    def step(self, state, action):
        if state == self.goal_state:
            return state, 0, True

        next_state = state
        if action == 0 and state > 2:
            next_state -= 3
        elif action == 1 and state < 3:
            next_state += 3
        elif action == 2 and state % 3 != 0:
            next_state -= 1
        elif action == 3 and state % 3 != 2:
            next_state += 1

        if next_state == self.goal_state:
            return next_state, 50, True
        else:
            return next_state, -1, False

In [7]:
def mc_control(env, episodes=5000, epsilon=0.1, gamma=1.0):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    returns_sum = defaultdict(lambda: np.zeros(len(env.actions)))
    returns_count = defaultdict(lambda: np.zeros(len(env.actions)))

    for i in range(episodes):
        episode = []
        state = random.randint(1, 5)
        done = False

        while not done:
            if random.random() < epsilon:
                action = random.choice(env.actions)
            else:
                action = np.argmax(Q[state])

            next_state, reward, done = env.step(state, action)
            episode.append((state, action, reward))
            state = next_state

        visited_state_actions = set()
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = gamma * G + r

            if (s, a) not in visited_state_actions:
                visited_state_actions.add((s, a))
                returns_sum[s][a] += G
                returns_count[s][a] += 1
                Q[s][a] = returns_sum[s][a] / returns_count[s][a]

    return Q

In [8]:
env = SimpleGridWorld()
optimal_q = mc_control(env, episodes=10000)

print("Calculated Optimal Action-Values (Q-Table):")
for state in range(1, 6):
    print(f"State S{state}: Up: {optimal_q[state][0]:.2f}, Down: {optimal_q[state][1]:.2f}, "
          f"Left: {optimal_q[state][2]:.2f}, Right: {optimal_q[state][3]:.2f}")

print("\nOptimal Policy (Best Action for each state):")
actions_map = {0: "UP", 1: "DOWN", 2: "LEFT", 3: "RIGHT"}
for state in range(1, 6):
    best_action = np.argmax(optimal_q[state])
    print(f"S{state} -> {actions_map[best_action]}")

Calculated Optimal Action-Values (Q-Table):
State S1: Up: 48.88, Down: 46.72, Left: 50.00, Right: 47.32
State S2: Up: 47.37, Down: 46.76, Left: 48.89, Right: 47.08
State S3: Up: 50.00, Down: 48.88, Left: 48.88, Right: 47.87
State S4: Up: 47.16, Down: 47.84, Left: 48.94, Right: 45.76
State S5: Up: 46.23, Down: 45.63, Left: 47.86, Right: 46.30

Optimal Policy (Best Action for each state):
S1 -> LEFT
S2 -> LEFT
S3 -> UP
S4 -> LEFT
S5 -> LEFT
